In [ ]:
"""
Objetivo: Processar os microdados abertos do CFPB/NMDB, padronizar a 
inadimplência hipotecária por estado e renderizar os mapas coropléticos.
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import sys
import os

warnings.filterwarnings('ignore')

# ==========================================
# 1. PROCESSAMENTO DE DADOS (CFPB/NMDB)
# ==========================================
def processar_dados_espaciais(arquivo_bruto: str, arquivo_limpo: str):
    print(f"[*] Processando painel de microdados do CFPB ({arquivo_bruto})...")
    
    try:
        df_raw = pd.read_csv(arquivo_bruto)
        colunas_necessarias = ['RegionType', 'Name', '2008-09']
        
        for col in colunas_necessarias:
            if col not in df_raw.columns:
                raise ValueError(f"Coluna estrutural ausente: {col}")
                
        df_estados = df_raw[df_raw['RegionType'] == 'State'].copy()
        
        mapa_estados = {
            'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
            'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC',
            'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL',
            'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA',
            'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
            'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
            'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
            'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
            'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
            'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
            'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
        }
        
        df_estados['State_Code'] = df_estados['Name'].map(mapa_estados)
        df_clean = df_estados[['State_Code', '2008-09']].rename(columns={'2008-09': 'inad_hipotecas_90d'}).dropna(subset=['State_Code'])
        
        df_clean.to_csv(arquivo_limpo, index=False)
        print(f"[*] Base processada. Dimensão: {df_clean.shape[0]} estados mapeados.")
        
    except FileNotFoundError:
        print(f"[ERRO] Arquivo '{arquivo_bruto}' não encontrado.")
        sys.exit(1)

# ==========================================
# 2. RENDERIZAÇÃO DO MAPA DE CALOR
# ==========================================
def renderizar_mapas(arquivo_limpo: str):
    print("\n[*] Renderizando mapas espaciais...")
    
    if not os.path.exists(arquivo_limpo):
        print(f"[ERRO] Base limpa '{arquivo_limpo}' ausente.")
        sys.exit(1)
        
    df_spatial = pd.read_csv(arquivo_limpo)
    
    portfolios = {
        'inad_hipotecas_90d': ('Hipotecas (90+ dias)', 'Reds'),
    }
    
    mapas_exportados = 0
    for col, (nome, paleta) in portfolios.items():
        if col not in df_spatial.columns:
            continue
            
        print(f" > Processando: {nome}")
        
        # Camada 1: Mapa de Calor (Choropleth)
        fig = px.choropleth(
            df_spatial, locations='State_Code', locationmode="USA-states",
            color=col, color_continuous_scale=paleta, scope="usa",
            labels={col: 'Inadimplência Grave (%)'}
        )
        
        # Camada 2: Siglas dos Estados (Scattergeo)
        fig.add_trace(go.Scattergeo(
            locations=df_spatial['State_Code'], locationmode="USA-states",
            text=df_spatial['State_Code'], mode="text",
            textfont=dict(color="black", size=10, family="Arial Black"), showlegend=False
        ))
        
        # Layout Executivo
        fig.update_layout(
            geo=dict(bgcolor='rgba(0,0,0,0)', lakecolor='#e0f3ff', showland=True, landcolor='#f4f4f4'),
            margin=dict(l=0, r=0, t=20, b=0)
        )
        
        try:
            # Nome exato esperado no arquivo .tex
            arquivo_png = f'06_mapa_espacial_{col}.png'
            fig.write_image(arquivo_png, scale=3)
            mapas_exportados += 1
        except Exception as e:
            print(f"[!] Falha ao exportar PNG. Verifique a instalação do 'kaleido'. Erro: {e}")
            
    print(f"\n[SUCESSO] Pipeline Espacial Concluído. Mapas renderizados: {mapas_exportados}.")

if __name__ == "__main__":
    processar_dados_espaciais('cfpb_nmdb_mortgages_2008Q3.csv', 'base_espacial_estados.csv')
    renderizar_mapas('base_espacial_estados.csv')